# Optuna hyperparameter optimization

# Import libraries and set configs

In [ ]:
import sys

sys.path.append("..")
sys.path.append("../..")

import ast
import json
import numpy as np
import pandas as pd

import optuna


class CFG:
    # maximum number of simultaneously opened trades
    max_num_simult_trades = 100
    # significance level, that is used to conduct a t-test between 2 models
    optimize_alpha = 0.2
    n_repeats = 1
    n_folds = 8
    min_precision = 0.5

# Load the train data

In [ ]:
train_df = pd.read_pickle("data/train_df.pkl")

# all data for the last 90 days are test
test_date = train_df["time"].max() - pd.to_timedelta(90, unit="D")

profitable_hours_df = pd.read_csv("data/profitable_hours.csv")
latest = profitable_hours_df.iloc[-1]
buy_hours = ast.literal_eval(latest["profitable_buy_hours"])
sell_hours = ast.literal_eval(latest["profitable_sell_hours"])

buy_mask = (train_df["ttype"] == "buy") & (train_df["time"].dt.hour.isin(buy_hours))
sell_mask = (train_df["ttype"] == "sell") & (train_df["time"].dt.hour.isin(sell_hours))
train_df = train_df[buy_mask | sell_mask].reset_index(drop=True)

fi = pd.read_csv("model/feature_importance.csv")

### Functions for backtest

In [ ]:
slippage = 0.002
TP = CFG.cls_target_ratio_tp - slippage
SL = CFG.cls_target_ratio_sl + slippage

# Optimize

In [ ]:
from utils.optimization_utils import make_objective

with open("model/bybit_tickers.json", "r") as f:
    bybit_tickers = json.load(f)

df_optuna_more_info = pd.DataFrame(columns=["result", "backtest_result", "oof_conf_score",
                                            "profit_objects", "oof_conf_obj_num", "scores"])
df_optuna_more_info.to_csv("optuna/optuna_lgbm_info.csv", index=False)

objective = make_objective(
    train_df, test_date, fi, bybit_tickers, TP, SL,
    n_folds=CFG.n_folds, optimize_alpha=CFG.optimize_alpha, min_precision=CFG.min_precision,
)
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=750)

print("Number of finished trials: {}".format(len(study.trials)))

print("Best trial:")
trial = study.best_trial

print("  Value: {}".format(trial.value))

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

df_optuna = study.trials_dataframe()
df_optuna = df_optuna.sort_values("value", ascending=False)
df_optuna.to_csv("optuna/optuna_lgbm.csv", index=False)

display(df_optuna.head(10))